# Stream a Response via our Cortex Platform

This example shows how to stream a response from the Snowflake Cortex platform using our platform client.

## Setup

In [1]:
from typing import cast

from utils import MockKernel, setup_notebook

from agent_platform.core.kernel import Kernel

if not setup_notebook():
    raise ValueError("Failed to setup notebook")

In [ ]:
from agent_platform.core.platforms.cortex import CortexPlatformParameters
from agent_platform.core.platforms.cortex.configs import CortexModelMap

# Configurations that will be used
default_model_map = CortexModelMap.get_default_instance()

# Platform Parameters
parameters = CortexPlatformParameters()

## Create a Platform Client

In [3]:
from agent_platform.core.platforms.cortex.client import CortexClient

cortex_client = CortexClient(
    parameters=parameters,
)

cortex_client.attach_kernel(cast(Kernel, MockKernel()))

## Create a Tool

In [4]:
from typing import Annotated

from agent_platform.core.tools import ToolDefinition


async def joke_judge(joke: Annotated[str, "A joke to judge"]) -> bool:
    """Judge if a joke is funny.

    Arguments:
        joke: A joke to judge.

    Returns:
        True if the joke is funny, False otherwise.
    """
    return False

joke_judge_tool = ToolDefinition.from_callable(joke_judge)

print(joke_judge_tool)

ToolDefinition(name='joke_judge', description='Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', input_schema={'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}, function=<function joke_judge at 0x7667ac7e11c0>)


## Create a Prompt

In [5]:
from agent_platform.core.prompts import PromptTextContent, PromptUserMessage
from agent_platform.core.prompts.prompt import Prompt

user_prompt = PromptUserMessage(
    content=[PromptTextContent(
        text="Please use the joke_judge tool to "
        "judge if the following joke is funny: "
        "\"Why did the chicken cross the road?\"\n"
        "\"To get to the other side!\"",
        )],
)
general_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt],
    tools=[joke_judge_tool],
)

await general_prompt.finalize_messages()
cortex_prompt = await cortex_client.converters.convert_prompt(general_prompt)

print(cortex_prompt)

CortexPrompt(messages=[CortexPromptMessage(role='system', content='You are a helpful assistant.', content_list=None), CortexPromptMessage(role='user', content='Please use the joke_judge tool to judge if the following joke is funny: "Why did the chicken cross the road?"\n"To get to the other side!"', content_list=[])], tools=[CortexPromptToolSpec(name='joke_judge', description='Judge if a joke is funny.\n\n    Arguments:\n        joke: A joke to judge.\n\n    Returns:\n        True if the joke is funny, False otherwise.', input_schema={'type': 'object', 'properties': {'joke': {'type': 'string', 'description': 'A joke to judge'}}, 'required': ['joke'], 'strict': True, 'additionalProperties': False}, type='generic')], max_tokens=4096, temperature=0.0, top_p=1.0)


## Request and Stream a Response

In [6]:
deltas = []
i = 0
async for delta in cortex_client.generate_stream_response(
    cortex_prompt, "claude-3-5-sonnet",
):
    print(f"CHUNK {i}: {delta!r}")
    deltas.append(delta)
    i += 1


CHUNK 0: GenericDelta(op='add', path='/content', value=[{'kind': 'text', 'text': 'I'}], from_=None)
CHUNK 1: GenericDelta(op='add', path='/usage', value={'input_tokens': 0, 'output_tokens': 0, 'total_tokens': 0}, from_=None)
CHUNK 2: GenericDelta(op='add', path='/role', value='agent', from_=None)
CHUNK 3: GenericDelta(op='concat_string', path='/content/0/text', value="'ll", from_=None)
CHUNK 4: GenericDelta(op='concat_string', path='/content/0/text', value=' help', from_=None)
CHUNK 5: GenericDelta(op='concat_string', path='/content/0/text', value=' you judge if', from_=None)
CHUNK 6: GenericDelta(op='concat_string', path='/content/0/text', value=' this', from_=None)
CHUNK 7: GenericDelta(op='concat_string', path='/content/0/text', value=' classic', from_=None)
CHUNK 8: GenericDelta(op='concat_string', path='/content/0/text', value=' joke is funny using', from_=None)
CHUNK 9: GenericDelta(op='concat_string', path='/content/0/text', value=' the joke_judge', from_=None)
CHUNK 10: Generic

## Reassemble

We now try to reassemble to full message to ensure it properly assembles into a `ResponseMessage`.

An implementation of this will need to be either in the AA or in a helper method attached to the kernel's `PlatformInterface`.

### First as a Dictionary

In [7]:
from pprint import pprint

from agent_platform.core.delta.combine_delta import combine_generic_deltas

assembeld_dict = combine_generic_deltas(deltas)
pprint(assembeld_dict)

{'content': [{'kind': 'text',
              'text': "I'll help you judge if this classic joke is funny using "
                      "the joke_judge tool. I'll combine both parts of the "
                      'joke into one string for evaluation.'},
             {'kind': 'tool_use',
              'tool_call_id': 'tooluse_itvvTPMQQh-o1qr-rF8Miw',
              'tool_input_raw': '{"joke": "Why did the chicken cross the road? '
                                'To get to the other side!"}',
              'tool_name': 'joke_judge'}],
 'metadata': {'sema4ai_metadata': {'platform_name': 'cortex'}},
 'role': 'agent',
 'usage': {'input_tokens': 457, 'output_tokens': 76, 'total_tokens': 533}}


### Now as a ResponseMessage

In [8]:
from agent_platform.core.responses import ResponseMessage

response_message = ResponseMessage.model_validate(assembeld_dict)
pprint(response_message)




ResponseMessage(content=[ResponseTextContent(kind='text',
                                             text="I'll help you judge if this "
                                                  'classic joke is funny using '
                                                  "the joke_judge tool. I'll "
                                                  'combine both parts of the '
                                                  'joke into one string for '
                                                  'evaluation.'),
                         ResponseToolUseContent(kind='tool_use',
                                                tool_call_id='tooluse_itvvTPMQQh-o1qr-rF8Miw',
                                                tool_name='joke_judge',
                                                tool_input_raw='{"joke": "Why '
                                                               'did the '
                                                               'chicken cross '
            

# Continue the Conversation

## Execute the Tool

In [9]:
from agent_platform.core.responses import ResponseToolUseContent

tool_use_content_block = response_message.content[1]
assert isinstance(tool_use_content_block, ResponseToolUseContent)

is_funny = await joke_judge(tool_use_content_block.tool_input["joke"])

print(f"Is the joke funny? {'Yes' if is_funny else 'No'}")

Is the joke funny? No


## Create a ToolResponse Content Block

> **Note:** This skips writing to a thread, but we convert the tool content block of the `ResponseMessage` to a `ThreadToolUsageContent` so we can convert to `PromptToolResultContent`.

In [10]:
from agent_platform.core.prompts import PromptTextContent, PromptToolResultContent

assert isinstance(tool_use_content_block, ResponseToolUseContent)
tool_result_content = PromptToolResultContent(
    tool_name=tool_use_content_block.tool_name,
    tool_call_id=tool_use_content_block.tool_call_id,
    content=[PromptTextContent(text=str(is_funny))],
)

print(tool_result_content)


PromptToolResultContent(kind='tool_result', tool_name='joke_judge', tool_call_id='tooluse_itvvTPMQQh-o1qr-rF8Miw', content=[PromptTextContent(kind='text', text='False')], is_error=False)


## Extract the AI's Tool Usage Content Block

We must send back the AI's original tool use in the new request

In [11]:
from agent_platform.core.prompts import PromptToolUseContent
from agent_platform.core.thread import ThreadToolUsageContent

original_tool_use = response_message.content[1]
assert isinstance(original_tool_use, ResponseToolUseContent)

# Convert to ThreadToolUsageContent and then to PromptToolUseContent
ai_tool_use = ThreadToolUsageContent.from_response_tool_use(original_tool_use)
prompt_tool_use = PromptToolUseContent.from_thread_tool_usage(ai_tool_use)

pprint(prompt_tool_use)


PromptToolUseContent(kind='tool_use',
                     tool_call_id='tooluse_itvvTPMQQh-o1qr-rF8Miw',
                     tool_name='joke_judge',
                     tool_input_raw='{"joke": "Why did the chicken cross the '
                                    'road? To get to the other side!"}',
                     _tool_input={'joke': 'Why did the chicken cross the road? '
                                          'To get to the other side!'})


## Create new Prompt

In [12]:
from agent_platform.core.prompts import PromptAgentMessage

return_user_prompt = PromptUserMessage(
    content=[tool_result_content],
)
return_tool_use = PromptAgentMessage(
    content=[prompt_tool_use],
)
return_gen_prompt = Prompt(
    system_instruction="You are a helpful assistant.",
    messages=[user_prompt, return_tool_use, return_user_prompt],
    tools=[joke_judge_tool],
)
await return_gen_prompt.finalize_messages()
return_cortex_prompt = await cortex_client.converters.convert_prompt(
    return_gen_prompt,
)

pprint(return_cortex_prompt)

CortexPrompt(messages=[CortexPromptMessage(role='system',
                                           content='You are a helpful '
                                                   'assistant.',
                                           content_list=None),
                       CortexPromptMessage(role='user',
                                           content='Please use the joke_judge '
                                                   'tool to judge if the '
                                                   'following joke is funny: '
                                                   '"Why did the chicken cross '
                                                   'the road?"\n'
                                                   '"To get to the other '
                                                   'side!"',
                                           content_list=[]),
                       CortexPromptMessage(role='assistant',
                                           cont

## Generate a new Response (Non-Streaming)

In [13]:
response = await cortex_client.generate_response(
    return_cortex_prompt,
    "claude-3-5-sonnet",
)

pprint(response)

ResponseMessage(content=[ResponseTextContent(kind='text',
                                             text='Based on the joke_judge '
                                                  "tool's evaluation, the "
                                                  'classic "Why did the '
                                                  'chicken cross the road?" '
                                                  'joke is not considered '
                                                  'funny. This is perhaps not '
                                                  'surprising, as this is one '
                                                  'of the most well-known and '
                                                  'basic jokes, often '
                                                  'considered more of an '
                                                  'anti-joke due to its '
                                                  'obvious and literal '
                           

In [14]:
# NOTE: something fishy is happening here, tool_calls are getting dropped non-streaming
# Reporting this to Snowflake to see what's up (perhaps my request structured is wrong)
response = await cortex_client.generate_response(
    cortex_prompt,
    "claude-3-5-sonnet",
)

pprint(response)

ResponseMessage(content=[ResponseTextContent(kind='text',
                                             text="I'll help you judge if this "
                                                  'classic joke is funny using '
                                                  'the joke_judge tool.')],
                role='agent',
                raw_response=None,
                stop_reason=None,
                usage=TokenUsage(input_tokens=457,
                                 output_tokens=62,
                                 total_tokens=519),
                metrics={},
                metadata={},
                additional_response_fields={})
